# Sports ratings
Our team, player and fighter ratings, loaded straight from the repo. Ratings by SportsDataHQ (sportsdatahq.com), CC BY 4.0.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sports-data-hq/sports-ratings/blob/main/ratings.ipynb)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/sports-data-hq/sports-ratings/main/ratings"

def load(sport, league, history=False):
    suffix = "_history" if history else ""
    return pd.read_csv(f"{BASE}/{sport}/{league}{suffix}.csv")

## Top 10 per league
Each file is one league's ratings today. Soccer, basketball and others use Elo (1500 is average); golf uses strokes per round versus the field.

In [ ]:
LEAGUES = [("basketball", "nba"), ("basketball", "wnba"), ("baseball", "mlb"),
           ("soccer", "premier_league"), ("tennis", "atp"), ("golf", "pga"),
           ("mma", "ufc"), ("cricket", "international")]

for sport, league in LEAGUES:
    try:
        df = load(sport, league)
    except Exception:
        print(f"{sport}/{league}: not available yet")
        continue
    # Files that hold several lists: pick one
    if league == "atp":
        df = df[df["variant"] == "overall"]
    if league == "international":
        df = df[(df["variant"] == "t20") & (df["group"] == "male")]
    top = df.sort_values("rating", ascending=False).head(10).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.barh(top["name"], top["rating"], color="#2b5d8a")
    ax.set_title(f"{league.upper()} top 10 ({df['date'].iloc[0]})")
    plt.tight_layout()
    plt.show()

## One team over time
The history file holds one block of rows per day. It starts on the first collection date, so the line fills in as days pass.

In [ ]:
hist = load("basketball", "nba", history=True)
team = hist["name"].iloc[0]   # change to any team name in the file
one = hist[hist["name"] == team].sort_values("date")
plt.figure(figsize=(7, 3))
plt.plot(pd.to_datetime(one["date"]), one["rating"], marker="o")
plt.title(f"{team}: rating by day")
plt.ylabel("Elo")
plt.tight_layout()
plt.show()
one.tail()

## Fresher or deeper data from the API
These files update once a day and hold one rating list per league. The API returns ratings for any date or season, plus schedules, standings, and team and player statistics. A free key gives 500 credits a month, no card needed. Each `get_ratings` call costs 2 credits.

Get a free key: [DunkAPI](https://dunkapi.com/signup?utm_source=github&utm_medium=readme&utm_campaign=sports-ratings), [StrikerAPI](https://strikerapi.com/signup?utm_source=github&utm_medium=readme&utm_campaign=sports-ratings), [AcesAPI](https://acesapi.com/signup?utm_source=github&utm_medium=readme&utm_campaign=sports-ratings) and the other brands listed in the [README](https://github.com/sports-data-hq/sports-ratings?utm_source=github&utm_medium=readme&utm_campaign=sports-ratings). A key works only on its own brand.

In [ ]:
import getpass, requests

key = getpass.getpass("DunkAPI key: ")
resp = requests.post(
    "https://mcp.dunkapi.com/v1/get_ratings",
    headers={"x-api-key": key, "content-type": "application/json"},
    json={"league": "nba", "season": 2025, "limit": 10},   # an earlier season
    timeout=30,
)
resp.raise_for_status()
pd.DataFrame(resp.json()["data"]["ratings"])